# colab-ollama

Jalankan Ollama di Google Colab + expose via Cloudflare Tunnel.

**Alur:**
1. Clone repo ke `/content`
2. Mount Google Drive (cache model antar recycle)
3. Jalankan `install.sh` → install Ollama + cloudflared, pull model, serve, tunnel
4. `install.sh` print URL akses luar + config ke `/tmp/colab-ollama.env`
5. Jalankan `keepalive.sh` di background → cegah idle timeout 90m

**Akses dari luar / Panrouter:**
```bash
curl $URL/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model":"qwen2.5:14b","messages":[{"role":"user","content":"halo"}]}'
```
OpenAI-compatible endpoint. Panrouter provider `ollama` tinggal arahkan base URL ke `$URL`.

**Model default:** `qwen2.5:14b` (patuh, minim sakti). Ganti via env: `MODEL=qwen2.5:7b bash install.sh`.


In [ ]:
# Clone repo
!git clone https://github.com/jhopan/colab-ollama /content/colab-ollama 2>/dev/null || echo 'sudah ada'

In [ ]:
# Mount Google Drive (cache model persist antar recycle)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install Ollama + cloudflared, pull model, serve, tunnel, print URL
# MODEL=qwen2.5:14b default; ganti sesuai mau
!cd /content/colab-ollama && bash install.sh

In [ ]:
# Keepalive: cegah idle timeout 90 menit
!nohup bash /content/colab-ollama/keepalive.sh > /tmp/keepalive.log 2>&1 &
print('keepalive di-start (background). Cek: tail /tmp/keepalive.log')

In [ ]:
# Print URL + test endpoint
import urllib.request, json as _j
env = open('/tmp/colab-ollama.env').read()
url = [l for l in env.splitlines() if l.startswith('OLLAMA_BASE_URL')][0].split('=',1)[1].rstrip()
print('BASE:', url)
try:
    r = urllib.request.urlopen(url + '/models', timeout=15)
    print('Model live:', _j.loads(r.read())['data'][0]['id'])
except Exception as e:
    print('Endpoint belum ready:', e)

## Catatan
- Colab idle timeout = 90 menit; hard limit 12-24 jam. Keepalive jaga dari idle, tapi recycle tetap ada.
- Pas recycle: notebook auto-run → install.sh rerun → Ollama fresh. Model di Drive tak pull ulang (cepat).
- GPU T4 (16GB): `qwen2.5:14b` jalan; 32b berat. CPU: pakai `qwen2.5:7b` atau `:3b`.
